# Chunk × SLM Distraction Audit — Phi-3.5-mini-instruct

**Model:** `microsoft/Phi-3.5-mini-instruct` (3.8B, bfloat16)  
**Datasets:** NQ-open · TriviaQA-rc · HotpotQA-distractor  
**Grid:** 3 datasets × 3 chunk sizes (128/256/512) × 2 retrievers (bm25/dense) + no-RAG baseline  
**= 21 configs** (7 per dataset)  

### Before running:
1. Set runtime to **GPU** (T4 or better): Runtime → Change runtime type → T4 GPU
2. Upload your **3 corpus JSONs** from your local `data/` folder to Google Drive at:
   ```
   My Drive/chunk_slm_audit/data/nq_corpus.json
   My Drive/chunk_slm_audit/data/triviaqa_corpus.json
   My Drive/chunk_slm_audit/data/hotpotqa_corpus.json
   ```
   *(These are pre-built locally — uploading avoids re-fetching Wikipedia for NQ)*
3. Add your HuggingFace token to Colab Secrets (🔑 icon in left sidebar) as `HF_TOKEN`

Results are **checkpointed to Drive after every config** — safe to reconnect if session times out.

In [ ]:
# ── Cell 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/chunk_slm_audit'
os.makedirs(f'{DRIVE_DIR}/data',    exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
print('✅ Drive mounted')
print(f'   Results will be saved to: {DRIVE_DIR}/results/')

In [ ]:
# ── Cell 2: Install Dependencies ──────────────────────────────────────────────
# Pin ranges tested against Colab's base environment (torch 2.x, Python 3.10+)
!pip install -q \
    "transformers>=4.44.0,<4.55.0" \
    "datasets>=2.20.0,<4.0.0" \
    "sentence-transformers>=2.7.0,<4.0.0" \
    "rank-bm25>=0.1.2" \
    "tiktoken>=0.7.0" \
    "wikipedia>=1.4.0" \
    "accelerate>=0.30.0" \
    "pandas>=2.0.0" \
    "tqdm"

print('✅ Dependencies installed')

In [ ]:
# ── Cell 3: HF Login & GPU Check ─────────────────────────────────────────────
from google.colab import userdata
import torch

# Load HF token from Colab Secrets
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('✅ HF_TOKEN loaded from Colab Secrets')
except Exception:
    HF_TOKEN = ''
    print('⚠️  No HF_TOKEN found in Secrets — Phi-3.5-mini is public so this may still work')

# Verify GPU
assert torch.cuda.is_available(), '❌ GPU not available — switch runtime to GPU!'
gpu_name = torch.cuda.get_device_name(0)
gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✅ GPU: {gpu_name}  ({gpu_mem:.1f} GB VRAM)')
print(f'   Phi-3.5-mini needs ~8 GB in bfloat16 — {"OK" if gpu_mem >= 8 else "WARNING: may OOM"}')
DEVICE = 'cuda'

In [ ]:
# ── Cell 4: Config (exact match with local experiment) ────────────────────────
import os

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

# ── Experiment grid — MUST match local config exactly ─────────────────────────
MODELS = {
    'Phi-3.5-mini': {
        'model_id': 'microsoft/Phi-3.5-mini-instruct',
        'dtype':    torch.bfloat16,
        'thinking': False,
    },
}

DATASETS    = ['nq', 'triviaqa', 'hotpotqa']   # outer loop
CHUNK_SIZES = [128, 256, 512]                   # tokens (tiktoken cl100k_base); 1024 removed
RETRIEVERS  = ['bm25', 'faiss']                 # 'faiss' label = dense cosine (numpy)
TOP_K       = 3
NUM_SAMPLES = 300

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR    = f'{DRIVE_DIR}/data'
RESULTS_DIR = f'{DRIVE_DIR}/results'
PARTIAL     = f'{RESULTS_DIR}/phi35_results_partial.csv'
FINAL       = f'{RESULTS_DIR}/phi35_results.csv'

print('Config loaded:')
print(f'  Model    : {list(MODELS.keys())}')
print(f'  Datasets : {DATASETS}')
print(f'  Chunks   : {CHUNK_SIZES}')
print(f'  Retrievers: {RETRIEVERS}  TOP_K={TOP_K}')
print(f'  Samples  : {NUM_SAMPLES}')
total = len(DATASETS) * len(MODELS) * (1 + len(CHUNK_SIZES) * len(RETRIEVERS))
print(f'  Total configs: {total}')

In [ ]:
# ── Cell 5: Data Preparation ──────────────────────────────────────────────────
import json
import re
import time
from pathlib import Path
from tqdm.auto import tqdm
from datasets import load_dataset

def prepare_nq(n=NUM_SAMPLES):
    out = Path(DATA_DIR) / 'nq_corpus.json'
    if out.exists():
        print('[data_prep] Loading cached NQ corpus from Drive …')
        return json.loads(out.read_text())
    # Fallback: build from scratch using Wikipedia API
    print('[data_prep] nq_corpus.json not found on Drive — fetching from HuggingFace + Wikipedia …')
    print('TIP: Upload nq_corpus.json from your local data/ folder to Drive to skip this step.')
    import wikipedia as _wp
    _wp.set_lang('en')

    def _fetch_wiki(question, answers):
        for q in [question] + list(answers[:2]):
            try:
                hits = _wp.search(q, results=4)
                for title in hits:
                    try:
                        page = _wp.page(title, auto_suggest=False)
                        if len(page.content.split()) >= 250:
                            return page.content
                    except (_wp.DisambiguationError, _wp.PageError):
                        continue
            except Exception:
                continue
        return None

    nq = load_dataset('google-research-datasets/nq_open', split='validation')
    samples, idx = [], 0
    with tqdm(total=n, desc='NQ: fetching Wikipedia articles') as pbar:
        while len(samples) < n and idx < len(nq):
            item = nq[idx]; idx += 1
            question, answers = item['question'], item['answer']
            if not answers:
                continue
            article = _fetch_wiki(question, answers)
            if article is None or len(article.split()) < 250:
                continue
            samples.append({'id': len(samples), 'dataset': 'nq',
                            'question': question, 'answers': answers, 'document': article})
            pbar.update(1)
            time.sleep(0.25)
    out.write_text(json.dumps(samples, indent=2, ensure_ascii=False))
    print(f'[data_prep] NQ: {len(samples)} samples saved to Drive')
    return samples


def prepare_triviaqa(n=NUM_SAMPLES):
    out = Path(DATA_DIR) / 'triviaqa_corpus.json'
    if out.exists():
        print('[data_prep] Loading cached TriviaQA corpus from Drive …')
        return json.loads(out.read_text())
    print('[data_prep] Loading TriviaQA rc validation split …')
    ds = load_dataset('mandarjoshi/trivia_qa', 'rc', split='validation',
                      trust_remote_code=True)
    samples = []
    with tqdm(total=n, desc='TriviaQA') as pbar:
        for item in ds:
            if len(samples) >= n:
                break
            answers = item['answer'].get('normalized_aliases') or \
                      [item['answer']['normalized_value']]
            answers = [a for a in answers if a.strip()]
            if not answers:
                continue
            wiki_contexts = [w for w in item.get('entity_pages', {}).get('wiki_context', [])
                             if len(w.split()) >= 50]
            if not wiki_contexts:
                continue
            document = '\n\n'.join(wiki_contexts)
            if len(document.split()) < 250:
                continue
            samples.append({'id': len(samples), 'dataset': 'triviaqa',
                            'question': item['question'], 'answers': answers,
                            'document': document})
            pbar.update(1)
    out.write_text(json.dumps(samples, indent=2, ensure_ascii=False))
    print(f'[data_prep] TriviaQA: {len(samples)} samples saved to Drive')
    return samples


def prepare_hotpotqa(n=NUM_SAMPLES):
    out = Path(DATA_DIR) / 'hotpotqa_corpus.json'
    if out.exists():
        print('[data_prep] Loading cached HotpotQA corpus from Drive …')
        return json.loads(out.read_text())
    print('[data_prep] Loading HotpotQA distractor validation split …')
    ds = load_dataset('hotpotqa/hotpot_qa', 'distractor', split='validation',
                      trust_remote_code=True)
    samples = []
    with tqdm(total=n, desc='HotpotQA') as pbar:
        for item in ds:
            if len(samples) >= n:
                break
            answer = item['answer'].strip()
            if not answer:
                continue
            titles     = item['context']['title']
            sent_lists = item['context']['sentences']
            paragraphs = [t + '. ' + ' '.join(s)
                          for t, s in zip(titles, sent_lists)]
            document = '\n\n'.join(paragraphs)
            if len(document.split()) < 100:
                continue
            samples.append({'id': len(samples), 'dataset': 'hotpotqa',
                            'question': item['question'], 'answers': [answer],
                            'document': document})
            pbar.update(1)
    out.write_text(json.dumps(samples, indent=2, ensure_ascii=False))
    print(f'[data_prep] HotpotQA: {len(samples)} samples saved to Drive')
    return samples


def prepare_dataset(name, n=NUM_SAMPLES):
    return {'nq': prepare_nq, 'triviaqa': prepare_triviaqa,
            'hotpotqa': prepare_hotpotqa}[name](n=n)

print('✅ Data prep functions loaded')

In [ ]:
# ── Cell 6: Chunker ───────────────────────────────────────────────────────────
import tiktoken

_ENC = tiktoken.get_encoding('cl100k_base')

def chunk_text(text, chunk_size, overlap=0):
    """Tokenizer-aware chunking — exact match with local chunker.py."""
    tokens = _ENC.encode(text)
    step   = chunk_size - overlap
    chunks = []
    for i in range(0, len(tokens), step):
        sl = tokens[i : i + chunk_size]
        if len(sl) < 10:
            break
        chunks.append(_ENC.decode(sl))
    return chunks

print('✅ Chunker loaded')

In [ ]:
# ── Cell 7: Retrievers ────────────────────────────────────────────────────────
# Dense retrieval uses numpy cosine (same as local) for result consistency.
# BM25Okapi for sparse retrieval.
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

_EMBED_MODEL = None

def _get_embed_model():
    global _EMBED_MODEL
    if _EMBED_MODEL is None:
        _EMBED_MODEL = SentenceTransformer(
            'sentence-transformers/all-MiniLM-L6-v2', device='cpu'
        )
    return _EMBED_MODEL


class BM25Retriever:
    def __init__(self, chunks):
        self.chunks = chunks
        self.bm25   = BM25Okapi([c.lower().split() for c in chunks])

    def retrieve(self, query, top_k=TOP_K):
        scores  = self.bm25.get_scores(query.lower().split())
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [self.chunks[i] for i in top_idx]


class DenseRetriever:
    """all-MiniLM-L6-v2 embeddings + numpy cosine — matches local retriever.py."""
    def __init__(self, chunks):
        self.chunks = chunks
        model       = _get_embed_model()
        embs        = model.encode(chunks, batch_size=64,
                                   show_progress_bar=False,
                                   normalize_embeddings=True)
        self.embs   = embs.astype(np.float32)

    def retrieve(self, query, top_k=TOP_K):
        model   = _get_embed_model()
        q_emb   = model.encode([query], normalize_embeddings=True).astype(np.float32)
        scores  = self.embs @ q_emb.T
        top_idx = np.argsort(scores[:, 0])[::-1][:top_k]
        return [self.chunks[i] for i in top_idx]


def build_retriever(chunks, retriever_type):
    if retriever_type == 'bm25':
        return BM25Retriever(chunks)
    if retriever_type == 'faiss':   # label kept for CSV compatibility
        return DenseRetriever(chunks)
    raise ValueError(f'Unknown retriever: {retriever_type!r}')

print('✅ Retrievers loaded')

In [ ]:
# ── Cell 8: Metrics ───────────────────────────────────────────────────────────
import re
import string

def _normalize(s):
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = s.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(s.split())

def exact_match(pred, golds):
    p = _normalize(pred)
    return float(any(p == _normalize(g) for g in golds))

def token_f1(pred, golds):
    p_toks = _normalize(pred).split()
    best   = 0.0
    for g in golds:
        g_toks = _normalize(g).split()
        if not p_toks or not g_toks:
            continue
        common = set(p_toks) & set(g_toks)
        if not common:
            continue
        prec = len(common) / len(p_toks)
        rec  = len(common) / len(g_toks)
        best = max(best, 2 * prec * rec / (prec + rec))
    return best

def distraction_ratio(baseline_em, rag_em):
    if baseline_em <= 1e-9:
        return 0.0
    return max(0.0, (baseline_em - rag_em) / baseline_em)

print('✅ Metrics loaded')

In [ ]:
# ── Cell 9: Model Runner ──────────────────────────────────────────────────────
# Phi-3.5-mini fix: do NOT use trust_remote_code=True for the MODEL.
# The custom modeling_phi3.py on HF Hub references DynamicCache.seen_tokens
# which was removed in transformers >= 4.45. Instead, use transformers' built-in
# Phi3ForCausalLM (auto-selected from config.json's "model_type": "phi3").
# The tokenizer can keep trust_remote_code=True — it's harmless and forward-compatible.

import gc
import shutil
from pathlib import Path
from typing import Optional
from transformers import AutoModelForCausalLM, AutoTokenizer

# Clear any cached custom phi3 code so it's never imported on subsequent loads
_phi3_cache = Path.home() / '.cache/huggingface/modules/transformers_modules/microsoft'
if _phi3_cache.exists():
    shutil.rmtree(_phi3_cache, ignore_errors=True)
    print(f'[setup] Cleared stale custom Phi3 module cache')

_model_cache     = {}
_tokenizer_cache = {}

def _evict_all():
    for k in list(_model_cache):
        del _model_cache[k]
        del _tokenizer_cache[k]
    gc.collect()
    torch.cuda.empty_cache()


def load_model(model_name):
    if model_name in _model_cache:
        return _model_cache[model_name], _tokenizer_cache[model_name]
    _evict_all()
    cfg      = MODELS[model_name]
    model_id = cfg['model_id']
    dtype    = cfg['dtype']
    print(f'\n[runner] Loading {model_name} ({model_id}) on {DEVICE} …')

    tok = AutoTokenizer.from_pretrained(
        model_id, token=HF_TOKEN or None, trust_remote_code=True
    )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    # trust_remote_code=False → use transformers' built-in Phi3ForCausalLM
    # (compatible with current DynamicCache API)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype,
        token=HF_TOKEN or None,
        trust_remote_code=False,          # ← use built-in implementation
        attn_implementation='eager',      # ← avoids flash_attn dependency
    ).to(DEVICE)
    mdl.eval()

    _model_cache[model_name]     = mdl
    _tokenizer_cache[model_name] = tok
    return mdl, tok


def build_prompt(model_name, question, context, tokenizer):
    if context:
        user = (f'Context:\n{context}\n\n'
                f'Question: {question}\n'
                'Answer in a short phrase (1-5 words):')
    else:
        user = (f'Question: {question}\n'
                'Answer in a short phrase (1-5 words):')
    messages = [{'role': 'user', 'content': user}]
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        return user + '\n'


_THINK_RE = re.compile(r'<think>.*?</think>', re.DOTALL)

def generate_answer(model_name, question, context=None):
    mdl, tok = load_model(model_name)
    prompt   = build_prompt(model_name, question, context, tok)
    inputs   = tok(prompt, return_tensors='pt', truncation=True,
                   max_length=2048).to(DEVICE)
    with torch.no_grad():
        out = mdl.generate(
            **inputs, max_new_tokens=50,
            do_sample=False, pad_token_id=tok.eos_token_id,
        )
    new_ids = out[0][inputs['input_ids'].shape[1]:]
    raw     = tok.decode(new_ids, skip_special_tokens=True).strip()
    raw     = _THINK_RE.sub('', raw).strip()
    for line in raw.splitlines():
        line = line.strip()
        if line:
            return line
    return raw

print('✅ Model runner loaded')

In [ ]:
# ── Cell 10: Resume Helpers & Drive Checkpoint ────────────────────────────────
import pandas as pd
from pathlib import Path

def _flush_memory():
    gc.collect()
    torch.cuda.synchronize()
    torch.cuda.empty_cache()


def _load_done():
    """Merge FINAL + PARTIAL to get done keys — same logic as local experiment.py."""
    frames = []
    for p in (FINAL, PARTIAL):
        if Path(p).exists():
            frames.append(pd.read_csv(p))
    if not frames:
        return set(), []
    df   = pd.concat(frames, ignore_index=True).drop_duplicates(
        subset=['dataset', 'model', 'chunk_size', 'retriever']
    )
    keys = {(r['dataset'], r['model'], str(r['chunk_size']), r['retriever'])
            for _, r in df.iterrows()}
    return keys, df.to_dict('records')


def _save(results):
    """Write partial results to Drive — checkpoint after every config."""
    pd.DataFrame(results).to_csv(PARTIAL, index=False)


print('✅ Resume helpers loaded')
done, existing = _load_done()
print(f'   Already done: {len(done)} configs (will be skipped)')

In [ ]:
# ── Cell 11: Experiment Loop ──────────────────────────────────────────────────
from tqdm.auto import tqdm

def run_baseline(dataset_name, model_name, samples):
    em_list, f1_list = [], []
    for s in tqdm(samples, desc=f'  [no-RAG] {model_name}', leave=False):
        pred = generate_answer(model_name, s['question'], context=None)
        em_list.append(exact_match(pred, s['answers']))
        f1_list.append(token_f1(pred, s['answers']))
    return {
        'dataset': dataset_name, 'model': model_name,
        'chunk_size': 'none', 'retriever': 'no_rag',
        'em':  round(sum(em_list) / len(em_list), 4),
        'f1':  round(sum(f1_list) / len(f1_list), 4),
        'distraction_ratio': 0.0, 'n_samples': len(samples),
    }


def run_rag(dataset_name, model_name, samples, chunk_size, retriever_type, baseline_em):
    _flush_memory()
    all_chunks = []
    for s in samples:
        all_chunks.extend(chunk_text(s['document'], chunk_size))
    print(f'  [index] {retriever_type} | cs={chunk_size} | {len(all_chunks)} chunks')
    retriever = build_retriever(all_chunks, retriever_type)

    em_list, f1_list = [], []
    desc = f'  [{retriever_type}] {model_name} cs={chunk_size}'
    for idx, s in enumerate(tqdm(samples, desc=desc, leave=False)):
        try:
            context = '\n\n'.join(retriever.retrieve(s['question'], top_k=TOP_K))
            pred    = generate_answer(model_name, s['question'], context=context)
        except Exception as exc:
            print(f'\n  [WARN] sample {s["id"]} failed: {exc}')
            pred = ''
        em_list.append(exact_match(pred, s['answers']))
        f1_list.append(token_f1(pred, s['answers']))
        if (idx + 1) % 10 == 0:
            _flush_memory()

    del retriever, all_chunks
    _flush_memory()

    avg_em = sum(em_list) / len(em_list)
    avg_f1 = sum(f1_list) / len(f1_list)
    return {
        'dataset': dataset_name, 'model': model_name,
        'chunk_size': chunk_size, 'retriever': retriever_type,
        'em':  round(avg_em, 4),
        'f1':  round(avg_f1, 4),
        'distraction_ratio': round(distraction_ratio(baseline_em, avg_em), 4),
        'n_samples': len(samples),
    }


def run_experiment():
    done, results = _load_done()
    print(f'Resuming — {len(done)} configs already done, {len(results)} rows loaded')

    for dataset_name in DATASETS:
        print(f'\n{"="*60}')
        print(f'DATASET: {dataset_name.upper()}')
        print(f'{"="*60}')
        samples = prepare_dataset(dataset_name)[:NUM_SAMPLES]

        for model_name in MODELS:
            # ── baseline ──────────────────────────────────────────────────
            key_base = (dataset_name, model_name, 'none', 'no_rag')
            if key_base not in done:
                print(f'\n{"-"*60}')
                print(f'[{dataset_name}] Model: {model_name}')
                row = run_baseline(dataset_name, model_name, samples)
                results.append(row)
                done.add(key_base)
                _save(results)   # ← Drive checkpoint
                _flush_memory()
                print(f'  Baseline  EM={row["em"]:.3f}  F1={row["f1"]:.3f}  [saved to Drive]')
            else:
                print(f'\n[skip] Baseline done: [{dataset_name}] {model_name}')

            base_em = next(
                r['em'] for r in results
                if r['dataset']   == dataset_name
                and r['model']    == model_name
                and r['retriever'] == 'no_rag'
            )

            # ── RAG grid ──────────────────────────────────────────────────
            for cs in CHUNK_SIZES:
                for ret in RETRIEVERS:
                    key = (dataset_name, model_name, str(cs), ret)
                    if key in done:
                        print(f'  [skip] [{dataset_name}] {model_name} cs={cs} {ret}')
                        continue
                    row = run_rag(dataset_name, model_name, samples, cs, ret, base_em)
                    results.append(row)
                    done.add(key)
                    _save(results)   # ← Drive checkpoint
                    _flush_memory()
                    print(f'  [{dataset_name}] cs={cs:4d}  {ret:5s}  '
                          f'EM={row["em"]:.3f}  F1={row["f1"]:.3f}  '
                          f'DR={row["distraction_ratio"]:.3f}  [saved to Drive]')

    # ── Finalize ──────────────────────────────────────────────────────────
    df = pd.DataFrame(results)
    df.to_csv(FINAL, index=False)
    if Path(PARTIAL).exists():
        Path(PARTIAL).unlink()
    print(f'\n✅ DONE — {len(df)} rows saved to {FINAL}')
    return df

print('✅ Experiment loop ready')

In [ ]:
# ── Cell 12: RUN ──────────────────────────────────────────────────────────────
# Verify corpus JSONs are on Drive before starting
missing = [d for d in DATASETS
           if not Path(f'{DATA_DIR}/{d}_corpus.json').exists()]
if missing:
    print(f'⚠️  Missing corpus files on Drive: {[f"{d}_corpus.json" for d in missing]}')
    print('Upload them from your local data/ folder to Google Drive at:')
    print(f'  {DATA_DIR}/')
    print('Then re-run this cell.')
    print('\nNQ requires Wikipedia API fetching if not pre-built — this is slow (~30 min).')
    print('TriviaQA and HotpotQA will download from HuggingFace automatically (no Wikipedia needed).')
else:
    print('✅ All corpus files found on Drive')
    print('Starting experiment …\n')
    df = run_experiment()
    print(df.to_string())

In [ ]:
# ── Cell 13: Download Results ─────────────────────────────────────────────────
# Results are already on Drive. Also download directly to your local machine:
from google.colab import files
files.download(FINAL)

In [ ]:
# ── Cell 14: Quick Summary Table ──────────────────────────────────────────────
import pandas as pd
from pathlib import Path

src = FINAL if Path(FINAL).exists() else PARTIAL
df  = pd.read_csv(src)

print(f'Results from: {src}  ({len(df)} rows)\n')

for dataset in df['dataset'].unique():
    sub = df[df['dataset'] == dataset]
    print(f'{'='*60}')
    print(f'[{dataset.upper()}]  EM by chunk_size × retriever')
    print(f'{'='*60}')
    pivot = sub[sub['retriever'] != 'no_rag'].pivot_table(
        index='retriever', columns='chunk_size', values='em'
    )
    print(pivot.to_string())
    baseline = sub[sub['retriever'] == 'no_rag']['em'].values
    if len(baseline):
        print(f'  no-RAG baseline EM: {baseline[0]:.3f}')
    print()